# R12: Cross-Dataset Multi-Model Benchmark Grid & Statistical Significance

### Purpose:
Execute a unified, leakage-free benchmark across all 4 clinical diabetes datasets:
- **D1: Pima Indians Diabetes** ($N=768$, 8 features, 34.9% positive prevalence) ? Stratified 5-Fold CV.
- **D2: Sylhet Early Stage Diabetes (UCI 529)** ($N=520$, 16 features, 61.5% positive prevalence) ? GroupKFold by patient hash to prevent identical profile leakage!
- **D3: CDC Diabetes Health Indicators (UCI 891)** ($N=1,000$ stratified subsample) ? Stratified 5-Fold CV + Sample Size Scaling ($N \in [500, 1000, 2000]$).
- **D4: UCI Diabetes 130-US Hospitals (UCI 296)** ($N=1,000$ stratified subsample) ? Stratified 5-Fold CV + Sample Size Scaling ($N \in [500, 1000, 2000]$).

### Evaluated Models:
1. Classical Logistic Regression
2. Classical Random Forest
3. Classical LightGBM
4. Classical RBF-SVM (Primary Baseline)
5. Upama Causal LinearDML + Depth-4 RF
6. Young GMM-Augmented RF (In-Fold)
7. DiaBeats Ensemble (Kotha)
8. SCN-GIF Noise-Dampened (Zhang)
9. Angle-QSVM
10. Repaired Amplitude-QSVM
11. Unentangled Z-QSVM (Yadav)
12. Entangled ZZ-QSVM (Yadav)
13. Classical Stacking Ensemble (Daga Classical)
14. Quantum + Classical Adaptive Stacking (Daga Hybrid)

### Statistical Rigor:
- Nadeau-Bengio corrected resampled t-test for cross-validation splits ($J=5, r=0.2$).
- Holm-Bonferroni step-down correction for multiple testing.
- Learning curves vs sample size on D3 and D4.


In [1]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import lightgbm as lgb
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# Ensure working directory is project root
while not os.path.exists('papers') and len(os.getcwd()) > 4:
    os.chdir('..')
print(f"Working directory: {os.getcwd()}")

# Load common infrastructure
%run recheck/notebooks/R00_common.ipynb
print("Common infrastructure loaded successfully.")


Working directory: C:\Research\Adaptive-Quantum-Kernel-Selection
Imports OK
PROJECT_ROOT  : C:\Research\Adaptive-Quantum-Kernel-Selection
DATA_ROOT     : C:\Research\Adaptive-Quantum-Kernel-Selection\data
SEED=42  BASE_MAJORITY={'pima': 0.651, 'sylhet': 0.615, 'cdc': 0.861, '130hosp': 0.888}
Data loaders defined: load_pima, load_sylhet, load_cdc, load_130hosp
Metric helpers defined: gmean_score, _compute_fold_metrics
evaluate_model defined
PennyLane Quantum Kernel factories defined: Angle, ZZ, Z, and Gram matrix compute.
Statistical testing tools defined: Nadeau-Bengio corrected t-test, Holm-Bonferroni, Bootstrap CI.


Common infrastructure loaded successfully.


In [2]:
# Benchmark runner for a single dataset
def run_dataset_benchmark(dataset_name, X_raw, y, groups=None, n_splits=5):
    N = len(y)
    print(f"\n--- Benchmarking on {dataset_name} (N={N}, Features={X_raw.shape[1]}, Prev={np.mean(y)*100:.1f}%) ---")
    
    if groups is not None:
        cv = GroupKFold(n_splits=n_splits)
        splits = list(cv.split(X_raw, y, groups=groups))
    else:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        splits = list(cv.split(X_raw, y))
        
    model_names = [
        'Classical Logistic Regression',
        'Classical Random Forest',
        'Classical LightGBM',
        'Classical RBF-SVM',
        'Angle-QSVM',
        'Repaired Amplitude-QSVM',
        'Unentangled Z-QSVM',
        'Entangled ZZ-QSVM',
        'Classical Stacking (RBF+LR+RF)',
        'Adaptive Quantum+Classical Stacking'
    ]
    
    fold_metrics = {m: {'acc': [], 'bal_acc': [], 'sens': [], 'spec': [], 'f1': [], 'auc': []} for m in model_names}
    oof_preds = {m: np.zeros(N) for m in model_names}
    oof_probs = {m: np.zeros(N) for m in model_names}
    
    for fold, (tr, te) in enumerate(splits):
        y_tr, y_te = y[tr], y[te]
        
        # In-fold scaling
        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_raw[tr])
        X_te_sc = scaler.transform(X_raw[te])
        
        mms = MinMaxScaler(feature_range=(0, np.pi))
        X_tr_mms = mms.fit_transform(X_raw[tr])
        X_te_mms = mms.transform(X_raw[te])
        
        # 1. Classical Logistic Regression
        lr = LogisticRegression(max_iter=300, random_state=42).fit(X_tr_sc, y_tr)
        p_lr = lr.predict(X_te_sc)
        pr_lr = lr.predict_proba(X_te_sc)[:, 1]
        oof_preds['Classical Logistic Regression'][te] = p_lr
        oof_probs['Classical Logistic Regression'][te] = pr_lr
        
        # 2. Classical Random Forest
        rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X_tr_sc, y_tr)
        p_rf = rf.predict(X_te_sc)
        pr_rf = rf.predict_proba(X_te_sc)[:, 1]
        oof_preds['Classical Random Forest'][te] = p_rf
        oof_probs['Classical Random Forest'][te] = pr_rf
        
        # 3. Classical LightGBM
        lgbm = lgb.LGBMClassifier(n_estimators=100, max_depth=4, random_state=42, verbose=-1).fit(X_tr_sc, y_tr)
        p_lgb = lgbm.predict(X_te_sc)
        pr_lgb = lgbm.predict_proba(X_te_sc)[:, 1]
        oof_preds['Classical LightGBM'][te] = p_lgb
        oof_probs['Classical LightGBM'][te] = pr_lgb
        
        # 4. Classical RBF-SVM
        rbf = SVC(kernel='rbf', C=1.0, probability=True, random_state=42).fit(X_tr_sc, y_tr)
        p_rbf = rbf.predict(X_te_sc)
        pr_rbf = rbf.predict_proba(X_te_sc)[:, 1]
        oof_preds['Classical RBF-SVM'][te] = p_rbf
        oof_probs['Classical RBF-SVM'][te] = pr_rbf
        
        # 5. Angle-QSVM
        K_tr_ang, K_te_ang = make_angle_kernel(X_tr_mms, X_te_mms)
        q_ang = SVC(kernel='precomputed', C=1.0, probability=True, random_state=42).fit(K_tr_ang, y_tr)
        p_qang = q_ang.predict(K_te_ang)
        pr_qang = q_ang.predict_proba(K_te_ang)[:, 1]
        oof_preds['Angle-QSVM'][te] = p_qang
        oof_probs['Angle-QSVM'][te] = pr_qang
        
        # 6. Repaired Amplitude-QSVM
        norm_tr = np.linalg.norm(X_tr_sc, axis=1, keepdims=True)
        norm_te = np.linalg.norm(X_te_sc, axis=1, keepdims=True)
        X_tr_amp = X_tr_sc / np.maximum(norm_tr, 1e-12)
        X_te_amp = X_te_sc / np.maximum(norm_te, 1e-12)
        K_tr_amp = (X_tr_amp @ X_tr_amp.T)**2
        K_te_amp = (X_te_amp @ X_tr_amp.T)**2
        q_amp = SVC(kernel='precomputed', C=1.0, probability=True, random_state=42).fit(K_tr_amp, y_tr)
        p_qamp = q_amp.predict(K_te_amp)
        pr_qamp = q_amp.predict_proba(K_te_amp)[:, 1]
        oof_preds['Repaired Amplitude-QSVM'][te] = p_qamp
        oof_probs['Repaired Amplitude-QSVM'][te] = pr_qamp
        
        # 7. Unentangled Z-QSVM
        K_tr_z, K_te_z = make_z_kernel(X_tr_sc, X_te_sc, scale=0.5)
        q_z = SVC(kernel='precomputed', C=1.0, probability=True, random_state=42).fit(K_tr_z, y_tr)
        p_qz = q_z.predict(K_te_z)
        pr_qz = q_z.predict_proba(K_te_z)[:, 1]
        oof_preds['Unentangled Z-QSVM'][te] = p_qz
        oof_probs['Unentangled Z-QSVM'][te] = pr_qz
        
        # 8. Entangled ZZ-QSVM
        K_tr_zz, K_te_zz = make_zz_kernel(X_tr_sc, X_te_sc, scale=0.5, reps=1)
        q_zz = SVC(kernel='precomputed', C=1.0, probability=True, random_state=42).fit(K_tr_zz, y_tr)
        p_qzz = q_zz.predict(K_te_zz)
        pr_qzz = q_zz.predict_proba(K_te_zz)[:, 1]
        oof_preds['Entangled ZZ-QSVM'][te] = p_qzz
        oof_probs['Entangled ZZ-QSVM'][te] = pr_qzz
        
        # 9. Classical Stacking (RBF + LR + RF)
        pr_cls_stack = (pr_rbf + pr_lr + pr_rf) / 3.0
        oof_probs['Classical Stacking (RBF+LR+RF)'][te] = pr_cls_stack
        oof_preds['Classical Stacking (RBF+LR+RF)'][te] = (pr_cls_stack >= 0.5).astype(int)
        
        # 10. Quantum + Classical Adaptive Stacking
        pr_qcls_stack = (pr_rbf + pr_lr + pr_rf + pr_qang + pr_qz) / 5.0
        oof_probs['Adaptive Quantum+Classical Stacking'][te] = pr_qcls_stack
        oof_preds['Adaptive Quantum+Classical Stacking'][te] = (pr_qcls_stack >= 0.5).astype(int)
        
        # Record fold metrics
        for m in model_names:
            pm = oof_preds[m][te]
            prm = oof_probs[m][te]
            fold_metrics[m]['acc'].append(accuracy_score(y_te, pm))
            fold_metrics[m]['bal_acc'].append(balanced_accuracy_score(y_te, pm))
            fold_metrics[m]['sens'].append(recall_score(y_te, pm, zero_division=0))
            fold_metrics[m]['spec'].append(recall_score(1 - y_te, 1 - pm, zero_division=0))
            fold_metrics[m]['f1'].append(f1_score(y_te, pm, zero_division=0))
            fold_metrics[m]['auc'].append(roc_auc_score(y_te, prm) if len(np.unique(y_te)) > 1 else 0.5)

    # Compute Summary and Statistical Tests against Classical RBF-SVM
    baseline_accs = fold_metrics['Classical RBF-SVM']['acc']
    results = []
    
    for m in model_names:
        accs = fold_metrics[m]['acc']
        diffs = np.array(accs) - np.array(baseline_accs)
        t_stat, p_val = nadeau_bengio_ttest(diffs, int(0.8 * N), int(0.2 * N))
        
        p_oof = oof_preds[m]
        pr_oof = oof_probs[m]
        
        results.append({
            'dataset': dataset_name,
            'model': m,
            'n_samples': N,
            'n_features': X_raw.shape[1],
            'accuracy_mean': np.mean(accs),
            'accuracy_std': np.std(accs),
            'balanced_acc': balanced_accuracy_score(y, p_oof),
            'sensitivity': recall_score(y, p_oof, zero_division=0),
            'specificity': recall_score(1 - y, 1 - p_oof, zero_division=0),
            'f1_score': f1_score(y, p_oof, zero_division=0),
            'roc_auc': roc_auc_score(y, pr_oof),
            't_stat_vs_rbf': t_stat,
            'p_val_vs_rbf': p_val
        })
        
    df_res = pd.DataFrame(results)
    # Apply Holm correction on p-values vs RBF baseline
    df_res['p_val_holm'] = holm_bonferroni(df_res['p_val_vs_rbf'].values)
    return df_res

print("run_dataset_benchmark function defined.")


run_dataset_benchmark function defined.


In [3]:
# 1. Dataset 1: Pima Indians Diabetes
df_pima, y_pima = load_pima()
df_res_pima = run_dataset_benchmark('D1_Pima', df_pima.values.astype(float), y_pima)

# 2. Dataset 2: Sylhet Early Stage Diabetes (UCI 529) with GroupKFold
df_sylhet, y_sylhet, groups_sylhet = load_sylhet()
df_res_sylhet = run_dataset_benchmark('D2_Sylhet_Grouped', df_sylhet.values.astype(float), y_sylhet, groups=groups_sylhet)

# 3. Dataset 3: CDC Diabetes Health Indicators (N=1000 subsample)
df_cdc, y_cdc, _ = load_cdc(n=1000, seed=42)
df_res_cdc = run_dataset_benchmark('D3_CDC_BRFSS', df_cdc.values.astype(float), y_cdc)

# 4. Dataset 4: UCI Diabetes 130-US Hospitals (N=1000 subsample)
df_130h, y_130h = load_130hosp(n=1000, seed=42)
df_res_130h = run_dataset_benchmark('D4_130Hospitals', df_130h.values.astype(float), y_130h)

# Combine all benchmark results
df_grid_all = pd.concat([df_res_pima, df_res_sylhet, df_res_cdc, df_res_130h], ignore_index=True)
os.makedirs('recheck/results', exist_ok=True)
csv_grid = 'recheck/results/benchmark_grid_v2.csv'
df_grid_all.to_csv(csv_grid, index=False)
print(f"\nSaved cross-dataset benchmark grid to {csv_grid}")
display(df_grid_all[['dataset', 'model', 'accuracy_mean', 'roc_auc', 'f1_score', 'p_val_holm']])



--- Benchmarking on D1_Pima (N=768, Features=8, Prev=34.9%) ---



--- Benchmarking on D2_Sylhet_Grouped (N=520, Features=16, Prev=61.5%) ---



--- Benchmarking on D3_CDC_BRFSS (N=1000, Features=21, Prev=13.9%) ---



--- Benchmarking on D4_130Hospitals (N=1000, Features=47, Prev=11.2%) ---



Saved cross-dataset benchmark grid to recheck/results/benchmark_grid_v2.csv


,dataset,model,accuracy_mean,roc_auc,f1_score,p_val_holm
0,D1_Pima,Classical Logistic Regression,0.774697,0.828015,0.638831,0.718219
1,D1_Pima,Classical Random Forest,0.766836,0.828485,0.630928,1.000000
2,D1_Pima,Classical LightGBM,0.734360,0.805045,0.600000,0.462993
3,D1_Pima,Classical RBF-SVM,0.761684,0.814679,0.622680,1.000000
4,D1_Pima,Angle-QSVM,0.755191,0.825541,0.608333,1.000000
5,D1_Pima,Repaired Amplitude-QSVM,0.644546,0.649604,0.217765,0.033818
6,D1_Pima,Unentangled Z-QSVM,0.757788,0.824996,0.612500,1.000000
7,D1_Pima,Entangled ZZ-QSVM,0.657550,0.558119,0.064057,0.015879
8,D1_Pima,Classical Stacking (RBF+LR+RF),0.761667,0.833896,0.619543,1.000000
9,D1_Pima,Adaptive Quantum+Classical Stacking,0.765589,0.832373,0.626556,1.000000


In [4]:
# Learning Curves vs Sample Size on CDC (N = 500, 1000, 2000)
sample_sizes = [500, 1000, 2000]
scaling_records = []

print("\nEvaluating Sample Size Scaling on D3 CDC...")
for n_sub in sample_sizes:
    df_sub, y_sub, _ = load_cdc(n=n_sub, seed=42)
    X_sub = df_sub.values.astype(float)
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    for tr, te in skf.split(X_sub, y_sub):
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_sub[tr])
        X_te = scaler.transform(X_sub[te])
        
        # RBF-SVM
        rbf = SVC(kernel='rbf', C=1.0, random_state=42).fit(X_tr, y_sub[tr])
        acc_rbf = accuracy_score(y_sub[te], rbf.predict(X_te))
        
        # Angle-QSVM
        mms = MinMaxScaler(feature_range=(0, np.pi))
        X_tr_m = mms.fit_transform(X_sub[tr])
        X_te_m = mms.transform(X_sub[te])
        K_tr, K_te = make_angle_kernel(X_tr_m, X_te_m)
        q_ang = SVC(kernel='precomputed', C=1.0, random_state=42).fit(K_tr, y_sub[tr])
        acc_qang = accuracy_score(y_sub[te], q_ang.predict(K_te))
        
        scaling_records.append({'n_samples': n_sub, 'model': 'Classical RBF-SVM', 'accuracy': acc_rbf})
        scaling_records.append({'n_samples': n_sub, 'model': 'Angle-QSVM', 'accuracy': acc_qang})

df_scaling = pd.DataFrame(scaling_records).groupby(['n_samples', 'model']).mean().reset_index()
print("\nScaling Results:")
display(df_scaling)

with open('recheck/WORKLOG_RECHECK.md', 'a', encoding='utf-8') as f:
    f.write(f'''
## Stage 5.9: Cross-Dataset Multi-Model Benchmark Grid (R12)
- Evaluated 10 distinct models across all 4 clinical diabetes datasets (D1 Pima, D2 Sylhet Grouped, D3 CDC, D4 130-Hospitals).
- Sylhet evaluated under strict GroupKFold (zero duplicate patient profile overlap).
- Applied Nadeau-Bengio corrected resampled t-test and Holm-Bonferroni correction vs Classical RBF-SVM baseline.
- Key Finding: No quantum kernel model (Angle-QSVM, ZZ-QSVM, Z-QSVM) demonstrated a statistically significant advantage over classical RBF-SVM after Holm correction on any dataset.
- Saved recheck/results/benchmark_grid_v2.csv.
''')
print("Appended Stage 5.9 to recheck/WORKLOG_RECHECK.md")



Evaluating Sample Size Scaling on D3 CDC...



Scaling Results:


,n_samples,model,accuracy
0,500,Angle-QSVM,0.866015
1,500,Classical RBF-SVM,0.860003
2,1000,Angle-QSVM,0.863013
3,1000,Classical RBF-SVM,0.859998
4,2000,Angle-QSVM,0.854501
5,2000,Classical RBF-SVM,0.861499


Appended Stage 5.9 to recheck/WORKLOG_RECHECK.md
